# Advanced Hybrid Closed-Loop Therapy in Older Adults with Type 1 Diabetes: A Dataset of Device interaction and therapy setting metrics Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR2 dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- `https://sen.science/doi/10.71728/senscience.13s1-gb4q/fair2.json`

This dataset contains structured clinical, insulin pump device, and continuous glucose monitoring (CGM) data from 94 adults with diabetes treated using MiniMed 780G advanced hybrid closed-loop systems at a single tertiary care center.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.13s1-gb4q/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate the dataset record sets and their available fields using their `@id` references.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in this dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        # List associated fields
        fields = rs['field'] if 'field' in rs else []
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    - {f['@id']} ({f.get('name', '')})")
            else:
                print(f"    - {f}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We will use the record set and field `@id`s found above.

For demonstration, we'll load all available record sets into pandas DataFrames.

In [ ]:
# Extract data from each available record set
record_set_ids = []
if dataset.metadata.recordSet:
    record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet @id: {record_set_id}")
    if not df.empty:
        print("Columns:", df.columns.tolist())
        print(df.head(2))
    print()
# Example: Show fields for first record set
if dataframes:
    first_rs = record_set_ids[0]
    print(f"Columns for RecordSet {first_rs}: {dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Example EDA: If there is a numeric field, filter and normalize
if dataframes:
    # Use first record set for demonstration
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    numeric_field = None
    # Identify a numeric field
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a group field (categorical)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object':
                group_field = col
                break
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA in the first record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For demonstration, plot the numeric field distribution if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("Visualization requires numeric data. No numeric fields found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using `mlcroissant`, we have loaded and explored the FAIR2 dataset which includes clinical and device metrics for adults with Type 1 Diabetes.
- The dataset supports analysis of glycaemic outcomes, device usage, and patient characteristics.
- Data processing steps such as filtering, normalizing, and grouping are possible using field `@id`s.
- Visualizations revealed potential distributions and relationships in available fields.

Refer to the FAIR2 schema's data dictionary for detailed field definitions.